In [1]:
import numpy as np

def random_nonoverlapping_positions(N, Lx, Ly, min_dist, existing, max_tries):
  pos = []
  if existing is None:
    existing = []

  while len(pos) < N:
    tries = 0

    while True:
      trial = np.array([np.random.uniform(-Lx/2, Lx/2), np.random.uniform(-Ly/2, Ly/2)])

      # checking against all the points
      all_points = list(existing) + pos

      if all(np.linalg.norm(trial - p) >= min_dist for p in all_points):
        pos.append(trial)
        break

      tries += 1
      if tries > max_tries:
        raise RuntimeError("Too dense: reduce N or min_dist")

  return np.array(pos)


In [2]:
def build_system(N_fixed, N_moving, Lx, Ly, arrangement, min_dist, max_tries, v0):
  # for fixed dipoles

  if arrangement == "grid":
    n = int(np.ceil(np.sqrt(N_fixed)))
    xs = np.linspace(-Lx/2, Lx/2, n)
    ys = np.linspace(-Ly/2, Ly/2, n)

    grid = [np.array([x, y]) for x in xs for y in ys]
    pos_f = np.array(grid[:N_fixed])

  elif arrangement == "random":
    pos_f = random_nonoverlapping_positions(N_fixed, Lx, Ly, min_dist, [], max_tries)

  else:
    raise ValueError("Unknown arrangement")

  # fixed dipole directions
  theta_f = np.random.uniform(0, 2*np.pi, len(pos_f))
  mu_f = np.column_stack((np.cos(theta_f), np.sin(theta_f)))

  vel_f = np.zeros_like(pos_f)

  # moving dipoles
  pos_m = random_nonoverlapping_positions(N_moving, Lx, Ly, min_dist, pos_f, max_tries)

  theta_m = np.random.uniform(0, 2*np.pi, N_moving)
  mu_m = np.column_stack((np.cos(theta_m), np.sin(theta_m)))

  vel_m = np.random.uniform(-1,1, (N_moving, 2))*v0

  pos = np.vstack([pos_f, pos_m])
  mu  = np.vstack([mu_f, mu_m])
  vel = np.vstack([vel_f, vel_m])



  fixed_mask = np.array([True]*len(pos_f) + [False]*len(pos_m))

  return pos, mu, vel, fixed_mask

In [3]:
def dipole_force(r, mu_i, mu_j):
  # Computing dipole-dipole interaction force between two particles.
  rmag = np.linalg.norm(r)
  if rmag < 1e-6:
    return np.zeros(2)
  rhat = r/rmag
  muir = np.dot(mu_i, rhat)
  mujr = np.dot(mu_j, rhat)
  return (1.0 / rmag**4) * (muir * mu_j + mujr * mu_i + (np.dot(mu_i, mu_j) - 5*muir*mujr) * rhat)

In [9]:
def minimum_image(r, Lx, Ly):
    r[0] -= Lx * round(r[0] / Lx)
    r[1] -= Ly * round(r[1] / Ly)
    return r

In [10]:
def handle_head_on_reflection_active(
    positions, velocities, Lx, Ly, t, last_collision,
    collision_distance, collision_cooldown, theta_c
):
# detect and resolve head-on collisions between active particles.
  N = len(positions)
  cos_theta_c = np.cos(theta_c)

  for i in range(N):
    for j in range(i+1, N):
      # cooldown check
      if last_collision[i, j] >= 0 and t - last_collision[i, j] < collision_cooldown:
        continue
      # Distance check
      r = positions[j] - positions[i]
      r = minimum_image(r, Lx, Ly)
      dist = np.linalg.norm(r)
      if dist < 1e-12 or dist > collision_distance:
        continue

      rhat = r / dist
      vi, vj = velocities[i], velocities[j]

      # anti-parallel check (head-on)
      cos_angle = np.dot(vi, vj) / (np.linalg.norm(vi) * np.linalg.norm(vj))
      if cos_angle > -cos_theta_c:
        continue

      # approaching check
      if not (np.dot(vj-vi, rhat) < 0):
        continue

      #separate and reverse
      target_sep = collision_distance + 1e-3
      delta = target_sep - dist
      shift = 0.5 * delta * rhat
      positions[i] -= shift
      positions[j] += shift

      #Wrap periodically
      for k in [i, j]:
        positions[k, 0] = (positions[k, 0] + Lx/2) % Lx - Lx/2
        positions[k, 1] = (positions[k, 1] + Ly/2) % Ly - Ly/2

      # reverse velocities
      velocities[i] = -vi
      velocities[j] = -vj

      last_collision[i, j] = t
      last_collision[j, i] = t


In [11]:
def handle_collision_with_fixed(
    positions, velocities, fixed_mask, Lx, Ly, t, last_collision,
    collision_distance, collision_cooldown
):
    """
    Resolves collisions between a moving particle and a fixed particle.
    Fixed particle never moves. Moving particle reflects off it elastically.
    No head-on angle check needed — any contact counts as a collision.
    """
    N = len(positions)

    for i in range(N):
        for j in range(i + 1, N):

            # we only want one fixed and one moving
            # skip if both fixed or both moving
            if fixed_mask[i] == fixed_mask[j]:
                continue

            # cooldown check
            if last_collision[i, j] >= 0 and t - last_collision[i, j] < collision_cooldown:
                continue

            # distance check
            r    = positions[j] - positions[i]
            r    = minimum_image(r, Lx, Ly)
            dist = np.linalg.norm(r)
            if dist < 1e-12 or dist > collision_distance:
                continue

            rhat = r / dist
            last_collision[i, j] = t
            last_collision[j, i] = t

            # figure out which one is fixed and which is moving
            if fixed_mask[i] and not fixed_mask[j]:
                # i is fixed, j is moving
                fixed_pos  = positions[i]
                moving_idx = j
            else:
                # j is fixed, i is moving
                fixed_pos  = positions[j]
                moving_idx = i
                rhat       = -rhat   # flip so rhat always points fixed→moving

            v = velocities[moving_idx].copy()

            # reflect velocity off the surface normal (rhat)
            velocities[moving_idx] = v - 2 * np.dot(v, rhat) * rhat

            # push moving particle outside collision radius so they don't stick
            positions[moving_idx] = fixed_pos + rhat * (collision_distance + 1e-3)

            # wrap periodically (fixed particle never needs wrapping)
            positions[moving_idx, 0] = (positions[moving_idx, 0] + Lx/2) % Lx - Lx/2
            positions[moving_idx, 1] = (positions[moving_idx, 1] + Ly/2) % Ly - Ly/2


In [12]:
# ── parameters ────────────────────────────────────────────────────────────────

Lx, Ly             = 20.0, 20.0
steps              = 2000
dt                 = 0.01
v0                 = 1.0
mu_voltage         = 1.0
a0                 = 1.0
alpha_size         = 0.2
gamma              = 1.0
mass               = 1.0
collision_distance = 1.0
collision_cooldown = 10
theta_c            = np.pi / 6
min_dist           = 2.0
max_tries          = 10000
N_fixed            = 3
N_moving           = 10
arrangement        = "random"


In [13]:
def run_simulation(N_fixed, N_moving, Lx, Ly, arrangement,
                   min_dist, max_tries, v0, mu_voltage,
                   a0, alpha_size, gamma, mass,
                   collision_distance, collision_cooldown, theta_c,
                   steps, dt):

    # STEP 1: build initial state
    positions, dipoles, velocities, fixed_mask = build_system(
        N_fixed, N_moving, Lx, Ly, arrangement, min_dist, max_tries, v0
    )
    N = len(positions)

    # STEP 2: scale dipoles by mu_voltage and size
    sizes   = a0 * np.ones(N)
    dipoles = mu_voltage * sizes[:, None] * dipoles

    # STEP 3: moving particles start along ±x only
    velocities[~fixed_mask, 0] = np.random.choice([-1, 1], size=N_moving) * v0
    velocities[~fixed_mask, 1] = 0.0

    last_collision = -np.ones((N, N), dtype=int)

    # STEP 4: storage
    traj        = np.zeros((steps + 1, N, 2))
    vel_traj    = np.zeros((steps + 1, N, 2))
    dipole_traj = np.zeros((steps + 1, N, 2))

    traj[0]        = positions.copy()
    vel_traj[0]    = velocities.copy()
    dipole_traj[0] = dipoles.copy()

    # STEP 5: time loop
    for t in range(1, steps + 1):

        forces = np.zeros((N, 2))

        # dipole-dipole forces (all pairs)
        for i in range(N):
            for j in range(i + 1, N):
                if  t - last_collision[i, j] < collision_cooldown:
                    continue
                r = positions[j] - positions[i]
                r = minimum_image(r, Lx, Ly)
                F = dipole_force(r, dipoles[i], dipoles[j])
                forces[i] -= F
                forces[j] += F

        # anchoring force — moving particles only
        forces[~fixed_mask, 1] += -gamma * sizes[~fixed_mask] * velocities[~fixed_mask, 1]

        # velocity update — moving particles only
        velocities[~fixed_mask] += (forces[~fixed_mask] / mass) * dt

        # ── COLLISION STEP (two separate calls, clean and readable) ──────────

        # call 1: moving vs moving (head-on angle check applies)
        # pass only moving particle indices via slicing
        moving_positions  = positions[~fixed_mask]
        moving_velocities = velocities[~fixed_mask]
        moving_last_col   = last_collision[np.ix_(~fixed_mask, ~fixed_mask)]

        handle_head_on_reflection_active(
            moving_positions, moving_velocities,
            Lx, Ly, t,
            moving_last_col,
            collision_distance, collision_cooldown, theta_c
        )

        # write results back into full arrays
        positions[~fixed_mask]  = moving_positions
        velocities[~fixed_mask] = moving_velocities
        last_collision[np.ix_(~fixed_mask, ~fixed_mask)] = moving_last_col

        # call 2: moving vs fixed (simple elastic reflection, no angle check)
        handle_collision_with_fixed(
            positions, velocities, fixed_mask,
            Lx, Ly, t, last_collision,
            collision_distance, collision_cooldown
        )

        # ── END COLLISION STEP ───────────────────────────────────────────────

        # speed rescaling — moving particles only
        speeds = np.linalg.norm(velocities[~fixed_mask], axis=1, keepdims=True)
        speeds = np.where(speeds < 1e-8, 1.0, speeds)
        velocities[~fixed_mask] = v0 * velocities[~fixed_mask] / speeds

        # position update — moving particles only
        positions[~fixed_mask] += velocities[~fixed_mask] * dt

        # periodic boundaries — moving particles only
        positions[~fixed_mask, 0] = (positions[~fixed_mask, 0] + Lx/2) % Lx - Lx/2
        positions[~fixed_mask, 1] = (positions[~fixed_mask, 1] + Ly/2) % Ly - Ly/2

        # update sizes and dipoles — moving particles only
        tilt                 = np.abs(velocities[~fixed_mask, 1]) / v0
        sizes[~fixed_mask]   = a0 * (1 + alpha_size * tilt)
        dirs                 = velocities[~fixed_mask] / np.linalg.norm(
                                   velocities[~fixed_mask], axis=1, keepdims=True)
        dipoles[~fixed_mask] = mu_voltage * sizes[~fixed_mask, None] * dirs
        # dipoles[:N_fixed] never changes

        # store
        traj[t]        = positions.copy()
        vel_traj[t]    = velocities.copy()
        dipole_traj[t] = dipoles.copy()

    return traj, vel_traj, dipole_traj



In [14]:
traj, vel_traj, dipole_traj = run_simulation(
    N_fixed, N_moving, Lx, Ly, arrangement,
    min_dist, max_tries, v0, mu_voltage,
    a0, alpha_size, gamma, mass,
    collision_distance, collision_cooldown, theta_c,
    steps, dt
)

print("Done. traj shape:", traj.shape)  # (steps+1, N_fixed+N_moving, 2)

Done. traj shape: (2001, 13, 2)
